# Cobertura de ómnibus por zona y hora
1. Divide Montevideo en hexágonos (H3 resolución 9, ~0,1 km² cada uno).
2. Para cada hexágono, tipo de día y hora cuenta cuántos viajes de ómnibus distintos pasan
   por alguna parada a distancia caminable (el hexágono y sus vecinos, ~350 m).
3. Guarda la tabla gold `cobertura_zona_hora`.
4. Genera el mapa interactivo con reloj (`mapa_cobertura.html`) y lo muestra acá.

**Uso:** Run all.

In [ ]:
SILVER = "workspace.movilidad_silver"
DESTINOS = ["workspace.movilidad_gold.cobertura_zona_hora"]
RESOLUCION_H3 = 9      # 9 ≈ hexágonos de ~175 m de lado
ANILLOS = 1            # 1 = el hexágono y sus vecinos (~350 m caminando)
VIAJES_COBERTURA_PLENA = 6   # 6 ómnibus/hora (espera ~5 min) = azul pleno

In [ ]:
sql_cobertura = f"""
WITH paradas AS (
  SELECT codigo_parada, h3_longlatash3(lon, lat, {RESOLUCION_H3}) AS celda_parada
  FROM {SILVER}.paradas
),
alcance AS (   -- cada parada "cubre" su hexágono y los vecinos
  SELECT codigo_parada, explode(h3_kring(celda_parada, {ANILLOS})) AS celda
  FROM paradas
),
pasadas AS (   -- un viaje = una salida de una variante (variante + frecuencia)
  SELECT cod_ubic_parada AS codigo_parada, tipo_dia_normalizado AS tipo_dia,
         CAST(FLOOR(segundos_desde_medianoche / 3600) % 24 AS INT) AS hora,
         concat('V', cod_variante, '-', frecuencia) AS viaje, cod_variante
  FROM {SILVER}.horarios_por_parada
  WHERE segundos_desde_medianoche IS NOT NULL
  UNION ALL
  SELECT cod_ubic_parada, tipo_dia_normalizado,
         CAST(FLOOR(segundos_desde_medianoche / 3600) % 24 AS INT),
         concat('C', cod_circular, '-', cod_variante, '-', frecuencia), cod_variante
  FROM {SILVER}.horarios_circulares
  WHERE segundos_desde_medianoche IS NOT NULL
    AND tipo_dia_normalizado IN ('HABIL', 'SABADO', 'DOMINGO')
),
oferta AS (
  SELECT a.celda, p.tipo_dia, p.hora,
         COUNT(DISTINCT p.viaje)         AS viajes,
         COUNT(DISTINCT p.cod_variante)  AS variantes,
         COUNT(DISTINCT a.codigo_parada) AS paradas_en_alcance
  FROM pasadas p JOIN alcance a USING (codigo_parada)
  GROUP BY ALL
),
grilla AS (    -- todas las combinaciones, para que las horas sin servicio queden en 0
  SELECT c.celda, t.tipo_dia, h.hora
  FROM (SELECT DISTINCT celda FROM alcance) c
  CROSS JOIN (SELECT explode(array('HABIL', 'SABADO', 'DOMINGO')) AS tipo_dia) t
  CROSS JOIN (SELECT explode(sequence(0, 23)) AS hora) h
)
SELECT g.celda,
       h3_h3tostring(g.celda)                    AS celda_hex,
       h3_centeraswkt(g.celda)                   AS centro_wkt,
       g.tipo_dia,
       g.hora,
       COALESCE(o.viajes, 0)                     AS viajes_hora,
       COALESCE(o.variantes, 0)                  AS variantes,
       COALESCE(o.paradas_en_alcance, 0)         AS paradas_en_alcance,
       CASE WHEN o.viajes > 0 THEN ROUND(30.0 / o.viajes, 1) END AS espera_promedio_min,
       ROUND(LEAST(1.0, GREATEST(0.0,
             LN(GREATEST(COALESCE(o.viajes, 0), 1)) / LN({VIAJES_COBERTURA_PLENA}))), 3) AS indice_cobertura
FROM grilla g
LEFT JOIN oferta o ON g.celda = o.celda AND g.tipo_dia = o.tipo_dia AND g.hora = o.hora
"""

TABLA = None
for destino in DESTINOS:
    try:
        spark.sql(f"CREATE OR REPLACE TABLE {destino} AS {sql_cobertura}")
        TABLA = destino
        print("Tabla creada:", destino)
        break
    except Exception as e:
        print(f"No se pudo crear {destino}: {str(e)[:150]}")
assert TABLA, "No se pudo crear la tabla en ningún esquema."

spark.sql(f"COMMENT ON TABLE {TABLA} IS 'Cobertura de ómnibus por hexágono H3 res {RESOLUCION_H3}, tipo de día y hora. "
          f"indice_cobertura: 0 = sin servicio o ≤1 ómnibus/h, 1 = {VIAJES_COBERTURA_PLENA}+ ómnibus/h a ~350 m.'")

In [ ]:
# Resumen: cuánta zona está bien cubierta en cada hora de un día hábil
display(spark.sql(f"""
SELECT hora,
       COUNT(*)                                         AS hexagonos,
       ROUND(AVG(viajes_hora), 1)                       AS omnibus_hora_promedio,
       ROUND(100 * AVG(CASE WHEN indice_cobertura >= 0.5 THEN 1 ELSE 0 END), 1) AS pct_zona_bien_cubierta,
       ROUND(100 * AVG(CASE WHEN viajes_hora = 0 THEN 1 ELSE 0 END), 1)         AS pct_zona_sin_servicio
FROM {TABLA}
WHERE tipo_dia = 'HABIL'
GROUP BY hora ORDER BY hora
"""))

In [ ]:
import json

pdf = spark.sql(f"SELECT celda_hex, tipo_dia, hora, viajes_hora FROM {TABLA}").toPandas()
clave = {"HABIL": "H", "SABADO": "S", "DOMINGO": "D"}
datos = {}
for hexid, grupo in pdf.groupby("celda_hex"):
    fila = {"H": [0] * 24, "S": [0] * 24, "D": [0] * 24}
    for t, h, v in zip(grupo["tipo_dia"], grupo["hora"], grupo["viajes_hora"]):
        fila[clave[t]][int(h)] = int(v)
    datos[hexid] = fila["H"] + fila["S"] + fila["D"]
print(len(datos), "hexágonos para el mapa")

# El azul más oscuro del mapa = el nivel del 5% de zonas-hora con más servicio
con_servicio = pdf.loc[pdf["viajes_hora"] > 0, "viajes_hora"]
ESCALA_MAX = int(max(VIAJES_COBERTURA_PLENA * 2, con_servicio.quantile(0.95)))
print("Escala del mapa: 1 a", ESCALA_MAX, "ómnibus por hora")

In [ ]:
HTML = r"""<!doctype html>
<html lang="es"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Cobertura STM Montevideo</title>
<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.css">
<script src="https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js"></script>
<script src="https://unpkg.com/h3-js@4.1.0/dist/h3-js.umd.js"></script>
<style>
  html,body{height:100%;margin:0;font-family:system-ui,-apple-system,"Segoe UI",sans-serif;background:#10161e}
  #mapa{position:absolute;inset:0}
  .panel{position:absolute;z-index:1000;top:16px;left:16px;background:rgba(16,22,30,.92);color:#e8edf3;
         border-radius:12px;padding:16px 18px;width:290px;box-shadow:0 6px 24px rgba(0,0,0,.35)}
  .panel h1{font-size:15px;margin:0 0 2px;font-weight:600}
  .panel .sub{font-size:12px;color:#97a3b4;margin-bottom:14px}
  .reloj{font-size:44px;font-weight:700;font-variant-numeric:tabular-nums;letter-spacing:.02em;line-height:1}
  .reloj small{font-size:15px;color:#97a3b4;font-weight:500}
  .fila{display:flex;gap:8px;align-items:center;margin-top:12px}
  input[type=range]{flex:1;accent-color:#5a9bea}
  button{background:#233041;color:#e8edf3;border:1px solid #33445a;border-radius:7px;padding:6px 10px;
         font:inherit;font-size:13px;cursor:pointer}
  button.activo{background:#2b6cb0;border-color:#2b6cb0}
  button:focus-visible{outline:2px solid #5a9bea;outline-offset:2px}
  .dias{display:flex;gap:6px;margin-top:12px}
  .dias button{flex:1}
  .leyenda{margin-top:16px;font-size:12px;color:#97a3b4}
  .barra{height:10px;border-radius:5px;margin:6px 0 4px;
         background:linear-gradient(90deg,#67001f 0%,#67001f 6%,#d6604d 6%,#f4a582 22%,#fddbc7 36%,#d1e5f0 50%,#92c5de 64%,#4393c3 78%,#2166ac 89%,#053061 100%)}
  .extremos{display:flex;justify-content:space-between}
  .dato{margin-top:14px;font-size:13px;line-height:1.5}
  .dato b{font-variant-numeric:tabular-nums}
  @media (max-width:520px){.panel{left:8px;right:8px;top:8px;width:auto;padding:12px 14px}.reloj{font-size:34px}}
</style></head><body>
<div id="mapa"></div>
<div class="panel">
  <h1>Cobertura de ómnibus STM</h1>
  <div class="sub">Ómnibus por hora a ~350 m de cada zona · horario teórico</div>
  <div class="reloj" id="reloj">08:00 <small>a 09:00</small></div>
  <div class="fila">
    <button id="play" aria-label="Reproducir">▶</button>
    <input id="hora" type="range" min="0" max="23" step="1" value="8" aria-label="Hora del día">
  </div>
  <div class="dias">
    <button data-d="0" class="activo">Hábil</button>
    <button data-d="1">Sábado</button>
    <button data-d="2">Domingo</button>
  </div>
  <div class="leyenda">
    <div class="barra"></div>
    <div class="extremos"><span>0</span><span>1/h</span><span id="medio"></span><span>__MAX__+/h</span></div>
  </div>
  <div class="dato" id="resumen"></div>
</div>
<script>
const DATOS = __DATOS__;
const MAXV = __MAX__, FRECUENTE = __FRECUENTE__;
const stops = [[0,[214,96,77]],[.17,[244,165,130]],[.32,[253,219,199]],[.47,[209,229,240]],[.62,[146,197,222]],
               [.77,[67,147,195]],[.89,[33,102,172]],[1,[5,48,97]]];
function color(v){
  if(v===0) return "#67001f";
  const t = Math.min(1, Math.log(Math.max(v,1))/Math.log(MAXV));
  for(let i=1;i<stops.length;i++){
    if(t<=stops[i][0]){
      const [t0,c0]=stops[i-1],[t1,c1]=stops[i],k=(t-t0)/(t1-t0);
      return "rgb("+c0.map((c,j)=>Math.round(c+(c1[j]-c)*k)).join(",")+")";
    }
  }
  return "rgb(33,102,172)";
}
const mapa = L.map("mapa",{preferCanvas:true,zoomControl:false}).setView([-34.86,-56.17],12);
L.control.zoom({position:"bottomright"}).addTo(mapa);
const ESRI = "https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/";
L.tileLayer(ESRI + "World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
  {attribution:"Mapa &copy; Esri · Datos: Intendencia de Montevideo",maxZoom:16}).addTo(mapa);
mapa.createPane("etiquetas"); mapa.getPane("etiquetas").style.zIndex = 650; mapa.getPane("etiquetas").style.pointerEvents = "none";
L.tileLayer(ESRI + "World_Light_Gray_Reference/MapServer/tile/{z}/{y}/{x}",
  {pane:"etiquetas",maxZoom:16}).addTo(mapa);
const renderer = L.canvas({padding:.5});
const capas = [];
for(const hex in DATOS){
  const pol = L.polygon(h3.cellToBoundary(hex),{renderer,weight:.8,opacity:.62,fillOpacity:.62});
  pol._v = DATOS[hex];
  pol.bindTooltip(()=>{
    const v = pol._v[dia*24+hora];
    return v ? `<b>${v}</b> ómnibus en la hora<br>espera promedio ~${(30/v).toFixed(1)} min` : "Sin ómnibus en esta hora";
  },{sticky:true});
  capas.push(pol.addTo(mapa));
}
let hora=8, dia=0, timer=null;
const nombres=["día hábil","sábado","domingo"];
function pintar(){
  let buenas=0, sin=0;
  for(const p of capas){
    const v=p._v[dia*24+hora];
    const c=color(v);
    p.setStyle({fillColor:c,color:c});
    if(v===0) sin++; else if(v>=FRECUENTE) buenas++;
  }
  const hh=String(hora).padStart(2,"0"), h2=String((hora+1)%24).padStart(2,"0");
  document.getElementById("reloj").innerHTML=`${hh}:00 <small>a ${h2}:00</small>`;
  document.getElementById("resumen").innerHTML=
    `En ${nombres[dia]} a esta hora:<br><b>${(100*buenas/capas.length).toFixed(0)}%</b> de la zona con un ómnibus cada 10 min o menos<br>`+
    `<b>${(100*sin/capas.length).toFixed(0)}%</b> sin ningún ómnibus`;
}
document.getElementById("hora").addEventListener("input",e=>{hora=+e.target.value;pintar();});
document.querySelectorAll(".dias button").forEach(b=>b.addEventListener("click",()=>{
  dia=+b.dataset.d;
  document.querySelectorAll(".dias button").forEach(x=>x.classList.toggle("activo",x===b));
  pintar();
}));
document.getElementById("play").addEventListener("click",e=>{
  if(timer){clearInterval(timer);timer=null;e.target.textContent="▶";return;}
  e.target.textContent="❚❚";
  timer=setInterval(()=>{hora=(hora+1)%24;document.getElementById("hora").value=hora;pintar();},800);
});
document.getElementById("medio").textContent = Math.round(Math.sqrt(MAXV)) + "/h";
pintar();
</script></body></html>"""

html = (HTML.replace("__DATOS__", json.dumps(datos, separators=(",", ":")))
            .replace("__MAX__", str(ESCALA_MAX))
            .replace("__FRECUENTE__", str(VIAJES_COBERTURA_PLENA)))

# Guardar al lado del notebook para descargarlo y abrirlo en el navegador
try:
    ruta_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    ruta_html = "/Workspace" + ruta_nb.rsplit("/", 1)[0] + "/mapa_cobertura.html"
    with open(ruta_html, "w", encoding="utf-8") as f:
        f.write(html)
    print("Mapa guardado en:", ruta_html, f"({len(html)/1e6:.1f} MB)")
except Exception as e:
    print("No se pudo guardar el HTML:", e)

In [ ]:
displayHTML(f'<iframe srcdoc="{html.replace("&", "&amp;").replace(chr(34), "&quot;")}" style="width:100%;height:720px;border:0"></iframe>')